In [0]:
from pyspark.sql import functions as F

CATALOGO = "v_credit_medallion"
SILVER = f"{CATALOGO}.silver"
GOLD = f"{CATALOGO}.gold"

print("Camada Gold preparada.")
print(f"Origem: {SILVER}")
print(f"Destino: {GOLD}")

In [0]:


# Carregar as tabelas da camada Silver
chamados = spark.table(f"{SILVER}.chamados")
custos = spark.table(f"{SILVER}.custos")
motivos = spark.table(f"{SILVER}.base_motivos")
canais= spark.table(f"{SILVER}.canais")

# Consolidar os dados sem excluir chamados que não possuem custo
gold_chamados = (
    chamados
    .join(
        motivos.select(
            "id_motivo",
            "nome_motivo",
            "categoria",
            "criticidade"
        ),
        on="id_motivo",
        how="left"
    )
    .join(
        custos.select("id_chamado", "custo"),
        on="id_chamado",
        how="left"
    )
    .withColumn(
        "tem_custo",
        F.when(F.col("custo").isNotNull(), 1).otherwise(0)
    )
    .join(
        canais.select('id_canal','nome_canal','tipo_canal')
        ,
        on = 'id_canal',
        how ='left'
    ).withColumnRenamed('nome_canal','canal')
)


In [0]:

# Garantir que o schema Gold existe
spark.sql("CREATE SCHEMA IF NOT EXISTS v_credit_medallion.gold")

# Salvar a tabela analítica
(
    gold_chamados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("v_credit_medallion.gold.analise_chamados")
)

print("Tabela Gold criada com sucesso!")


In [0]:
tabela_gold = spark.table(
    "v_credit_medallion.gold.analise_chamados"
)

print("Total de registros:", tabela_gold.count())

display(tabela_gold.limit(10))

In [0]:

from pyspark.sql import functions as F

gold_indicadores = (
    tabela_gold
    .groupBy("categoria", "nome_motivo", "canal")
    .agg(
        F.countDistinct("id_chamado").alias("total_chamados"),

        F.sum(
            F.when(F.col("resolvido") == 1, 1).otherwise(0)
        ).alias("chamados_resolvidos"),

        F.avg(
            F.when(
                F.col("resolvido").isin(0, 1),
                F.col("resolvido")
            )
        ).alias("taxa_resolucao"),

        F.avg("tempo_espera").alias("tempo_medio_espera"),

        F.avg("tempo_atendimento").alias("tempo_medio_atendimento"),

        F.sum("custo").alias("custo_total_registrado"),

        F.avg("custo").alias("custo_medio_registrado"),

        F.sum(
            F.when(F.col("tem_custo") == 0, 1).otherwise(0)
        ).alias("chamados_sem_custo")
    )
    .withColumn(
        "taxa_resolucao_pct",
        F.round(F.col("taxa_resolucao") * 100, 2)
    )
    .withColumn(
        "tempo_medio_espera",
        F.round("tempo_medio_espera", 2)
    )
    .withColumn(
        "tempo_medio_atendimento",
        F.round("tempo_medio_atendimento", 2)
    )
    .withColumn(
        "custo_total_registrado",
        F.round("custo_total_registrado", 2)
    )
    .withColumn(
        "custo_medio_registrado",
        F.round("custo_medio_registrado", 2)
    )
)

display(
    gold_indicadores
    .orderBy(F.desc("total_chamados"))
)


In [0]:

#Persistência dos indicadores de atendimento

(
    gold_indicadores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.indicadores_atendimento")
)

print("Tabela de indicadores salva com sucesso.")


In [0]:

# Análise de satisfação por categoria, motivo e canal

pesquisas = spark.table(f"{SILVER}.pesquisa_satisfacao")

# Consolidar as pesquisas por chamado para evitar duplicações
pesquisa_por_chamado = (
    pesquisas
    .groupBy("id_chamado")
    .agg(
        F.avg("nota_atendimento").alias("nota_media"),
        F.count("id_pesquisa").alias("quantidade_pesquisas")
    )
)

# Relacionar satisfação aos dados consolidados da Gold
gold_satisfacao = (
    tabela_gold
    .join(pesquisa_por_chamado, on="id_chamado", how="left")
    .groupBy("categoria", "nome_motivo", "canal")
    .agg(
        F.countDistinct("id_chamado").alias("total_chamados"),
        F.count("nota_media").alias("chamados_com_avaliacao"),
        F.round(F.avg("nota_media"), 2).alias("nota_media_satisfacao"),
        F.round(
            F.avg(
                F.when(F.col("nota_media") >= 4, 1.0)
                 .when(F.col("nota_media").isNotNull(), 0.0)
            ) * 100,
            2
        ).alias("avaliacoes_positivas_pct")
    )
)

# Salvar a tabela Gold
(
    gold_satisfacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.indicadores_satisfacao")
)

display(
    spark.table(f"{GOLD}.indicadores_satisfacao")
    .orderBy(F.desc("total_chamados"))
)


In [0]:

# Painel executivo consolidado

indicadores_atendimento = spark.table(f"{GOLD}.indicadores_atendimento")
indicadores_satisfacao = spark.table(f"{GOLD}.indicadores_satisfacao")

# Remover a contagem duplicada para manter uma única coluna de volume
satisfacao_selecionada = indicadores_satisfacao.select(
    "categoria",
    "nome_motivo",
    "canal",
    "chamados_com_avaliacao",
    "nota_media_satisfacao",
    "avaliacoes_positivas_pct"
)

painel_executivo = indicadores_atendimento.join(
    satisfacao_selecionada,
    on=["categoria", "nome_motivo", "canal"],
    how="left"
)

(
    painel_executivo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.painel_executivo")
)

print("Painel executivo salvo na camada Gold.")
display(
    spark.table(f"{GOLD}.painel_executivo")
    .orderBy(F.desc("total_chamados"))
)


#### Preço médio e total por serviço

In [0]:
chamados = spark.table(f"{SILVER}.chamados")
custos = spark.table(f"{SILVER}.custos").select("id_chamado", "custo")
motivos_alta = (
    spark.table(f"{SILVER}.base_motivos")
    # .filter(F.col("criticidade") == "alta")
    .select("id_motivo", "nome_motivo", "criticidade", "categoria")
)

# Quantidade de chamados, preço médio e custo total por processo (motivo)
quantidade_por_motivo = (
    chamados
    .join(motivos_alta, on="id_motivo", how="inner")
    .join(custos, on="id_chamado", how="left")
    .groupBy("nome_motivo", "categoria", "criticidade")
    .agg(
        F.count("*").alias("qtd_chamados"),
        F.round(F.avg("custo"), 2).alias("custo_medio"),
        F.round(F.sum("custo"), 2).alias("custo_total"),
    )
    .orderBy(F.desc("qtd_chamados"))
)

(
quantidade_por_motivo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.custos_motivos")
)

quantidade_por_motivo.display()

#### Avaliação dos atendentes ativos

In [0]:
chamados = spark.table(f"{SILVER}.chamados")
atendentes = spark.table(f"{SILVER}.base_atendentes")
pesquisas = spark.table(f"{SILVER}.pesquisa_satisfacao")

# Uma nota média por chamado (evita duplicar chamados com múltiplas pesquisas)
nota_por_chamado = pesquisas.groupBy("id_chamado").agg(
    F.round(F.avg("nota_atendimento"), 2).alias("nota_chamado")
)

# Comparativo de atendentes: volume, satisfação média e avaliações positivas
# (join interno exclui ids sem cadastro, ex.: 0, 99, 999 - autoatendimento)
comparativo_atendentes = (
    chamados
    .join(nota_por_chamado, on="id_chamado", how="left")
    .join(atendentes, on="id_atendente", how="inner")
    .groupBy("id_atendente", "nome_atendente", "nivel_atendimento", "status_atendente")
    .agg(
        # F.countDistinct("id_chamado").alias("qtd_chamados"),
        # F.count("nota_chamado").alias("chamados_avaliados"),
        F.round(F.avg("nota_chamado"), 2).alias("nota_media_satisfacao"),
        F.round(
            F.avg(
                F.when(F.col("nota_chamado") >= 4, 1.0)
                 .when(F.col("nota_chamado").isNotNull(), 0.0)
            ) * 100,
            2
        ).alias("pct_positivas"),
    )
    .orderBy(F.desc("nota_media_satisfacao"))

    .filter(F.col("status_atendente") == "ativo")
)

comparativo_atendentes.display()